# g1_limpo — treino DO ZERO na Kaggle

Sem resume, sem checkpoint de entrada, sem conferência de paridade. O `smoke.py` é
quem confere a configuração; aqui só se treina.

**Antes de rodar:**

| | |
|---|---|
| Settings -> Accelerator | **GPU** |
| Settings -> Internet | **On** (o pip e o clone precisam) |
| Add-ons -> Secrets | `KAGGLE_USERNAME` e `KAGGLE_KEY`, os dois **anexados** |
| a branch | precisa estar **no GitHub** — o clone não vê o seu disco |

**Rode as células de cima a baixo.** A última sobe o checkpoint como versão nova do
dataset; sem ela a sessão morre e leva `/kaggle/working` junto.

⚠ A Kaggle corta em 12 h. Um treino do zero não cabe numa sessão: ele para na
`max_iterations` que couber no relógio, você sobe o checkpoint, e a **próxima** sessão
usa o notebook de RESUME (`g1_limpo_kaggle.ipynb`), não este.

In [ ]:
# ⚠ NADA DE `import torch` AQUI. O torch registra operadores C++ no import, e se ele
# entrar no kernel ANTES do pip, um reload depois levanta
# `Only a single TORCH_LIBRARY can be used to register the namespace triton`.
import subprocess, sys

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True)
print(smi.stdout or smi.stderr)
assert smi.returncode == 0 and smi.stdout.strip(), \
    "sem GPU. Settings -> Accelerator -> GPU"
print("python", sys.version.split()[0])

In [ ]:
import subprocess, sys

# ⚠⚠ LISTA DE ARGUMENTOS, NUNCA STRING DE SHELL. Com `!pip install ... numpy<2.5` o
# shell lê `<` como REDIRECIONAMENTO, tenta abrir um arquivo chamado `2.5`, aborta com
# exit 2 — e o pip NUNCA RODA. Com `-q` e sem conferir o código de saída, o erro só
# aparece duas células depois, como `No module named 'mjlab'`.
cmd = [sys.executable, "-m", "pip", "install", "--no-warn-conflicts", "mjlab==1.5.3"]
print(" ".join(cmd), flush=True)
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode != 0:
    print(r.stderr[-2000:])
assert r.returncode == 0, \
    "o pip falhou. Causa mais comum: internet DESLIGADA (Settings -> Internet -> On)"

In [ ]:
import subprocess, sys

# ⚠ EM SUBPROCESSO, e só depois no kernel: se o pip trocou o torch, o import aqui
# travaria a versão errada neste processo e não daria para desfazer sem reiniciar.
chk = subprocess.run(
    [sys.executable, "-c",
     "import torch;print(torch.__version__, torch.cuda.is_available())"],
    capture_output=True, text=True)
print("subprocesso:", chk.stdout.strip() or chk.stderr[-600:])
assert " True" in chk.stdout, \
    "o pip trocou o torch e a CUDA foi embora. Reinicie o kernel e rode da célula 1"

import torch, mjlab, mujoco, warp
print(f"torch {torch.__version__}  {torch.cuda.get_device_name(0)}")
print("mjlab", getattr(mjlab, "__version__", "?"),
      "| mujoco", mujoco.__version__, "| warp", warp.config.version)

In [ ]:
import importlib, os, pathlib, shutil, subprocess, sys

os.environ.setdefault("MUJOCO_GL", "egl")

RUN    = "bloco18"                 # ⚠ NOME NOVO POR BLOCO. Ele nomeia a pasta de run, o
                                   # zip e a mensagem do dataset. Reusar um nome mistura
                                   # duas tabelas de recompensa no mesmo log.
# ⚠ BRANCH EXPERIMENTAL. As correções de 14/09 vivem aqui; o pipeline estável está na
# tag `estavel-bloco17` e é o que o `g1_limpo_kaggle.ipynb` clona.
BRANCH = "exp/g1-limpo-v3"

BASE     = pathlib.Path("/kaggle/working")
RAIZ     = BASE / "g1"             # o clone, refeito a cada sessão
LOG_ROOT = BASE / "logs"           # ⚠ FORA de RAIZ: o re-clone apaga o que está dentro
raiz_exp = LOG_ROOT / "g1_limpo"   # <log_root>/<experiment_name>

if RAIZ.exists():
    shutil.rmtree(RAIZ)
subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--depth", "1",
                "https://github.com/JoaoBornelli/g1_training.git", str(RAIZ)],
               check=True)
print("clone =", subprocess.run(["git", "-C", str(RAIZ), "log", "--oneline", "-1"],
                                capture_output=True, text=True).stdout.strip())

# ⚠ `invalidate_caches` NÃO é higiene. O Python guarda um finder POR DIRETÓRIO em
# `sys.path_importer_cache`, e o finder de um diretório que não existia no momento da
# inserção fica cacheado como VAZIO — `import g1_limpo` falharia com `No module named`
# mesmo com o pacote em disco.
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
importlib.invalidate_caches()

print("run      =", RUN)
print("log_root =", LOG_ROOT)

In [ ]:
import pathlib, re, zipfile
from IPython.display import FileLink, display

SAIDA = BASE
ULTIMO_CKPT = ULTIMA_IT = ULTIMO_PACOTE = None


def _run_nova(run=RUN):
    """A pasta de run mais recente e seus `model_*.pt` por número — ou `(None, [])`."""
    if not raiz_exp.is_dir():
        return None, []
    runs = sorted(p for p in raiz_exp.iterdir()
                  if p.is_dir() and p.name.endswith(run))
    if not runs:
        return None, []
    cks = sorted(runs[-1].glob("model_*.pt"),
                 key=lambda p: int(re.search(r"(\d+)", p.name).group(1)))
    return runs[-1], cks


def empacota(run=RUN, baixa=True):
    """Zipa o ÚLTIMO checkpoint + os tfevents e oferece o download.

    ⚠ UM ZIP SÓ, e não os arquivos soltos. Um bloco longo deixa dezenas de
    checkpoints; baixar um por um é o que faz a sessão expirar no meio. E o tfevents
    vai junto porque sem ele o `leitura.py` não tem o que ler.
    """
    global ULTIMO_CKPT, ULTIMA_IT, ULTIMO_PACOTE
    nova, cks = _run_nova(run)
    if not cks:
        print("nenhum checkpoint ainda — o treino não salvou nada")
        return None
    ultimo = cks[-1]
    it = int(re.search(r"(\d+)", ultimo.name).group(1))
    pacote = SAIDA / f"{run}_it{it}.zip"
    with zipfile.ZipFile(pacote, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(ultimo, ultimo.name)
        for ev in sorted(nova.glob("events.out.tfevents*")):
            z.write(ev, ev.name)
        for p in sorted(nova.glob("params/*")):
            z.write(p, f"params/{p.name}")
    ULTIMO_CKPT, ULTIMA_IT, ULTIMO_PACOTE = ultimo, it, pacote
    print(f"{pacote}  ({pacote.stat().st_size / 2**20:.1f} MB)  it {it}")
    if baixa:
        display(FileLink(str(pacote.relative_to(BASE))))
    return pacote

In [ ]:
# =====================================================================
#  TREINO DO ZERO — sem resume, sem checkpoint de entrada
# =====================================================================
import dataclasses, sys
sys.path.insert(0, str(RAIZ))

import g1_limpo
from mjlab.scripts.train import TrainConfig, launch_training

# ⚠ 4096 É O NÚMERO DA KAGGLE, e ele NÃO vem da VRAM. Medido em 2026-09-11: a GPU do
# Colab tem 16 GB e roda 8192; a da Kaggle, também de 16 GB, não. Se der OOM: 2048.
NUM_ENVS = 4096

# ⚠ A META É ABSOLUTA e do zero não cabe numa sessão. A célula corta pelo relógio e
# imprime onde vai parar; a próxima sessão continua pelo notebook de RESUME.
META = 30000

# ⚠ O TETO DE PAREDE DA KAGGLE É DURO: 12 h, e a sessão morta leva `/kaggle/working`.
# O `SEG_POR_ITER` está ancorado em medição: 4096 envs num T4 deram 5,09 s/iter no
# g1_multitask (19 307 passos/s). Os 7,0 abaixo são margem para a cena do g1_limpo ser
# mais carregada — mesa, caixa, 29 termos de recompensa.
# ⚠ CORRIJA pelo `Collection time` real do primeiro log.
HORAS_LIMITE = 10.5
SEG_POR_ITER = 7.0

cfg = dataclasses.replace(TrainConfig.from_task(g1_limpo.TASK_ID),
                          log_root=str(LOG_ROOT))
cfg.env.scene.num_envs = NUM_ENVS
cfg.agent.run_name = RUN
cfg.agent.logger = "tensorboard"

# ⚠⚠ DO ZERO: `resume = False` e a LR fica no DEFAULT. O degrau para 5e-4 é regra de
# WARM-START — ele existe porque a função de valor de um checkpoint velho está errada
# numa distribuição nova. Aqui não há função de valor; baixar a LR só atrasaria.
cfg.agent.resume = False

cabe_tempo = int(HORAS_LIMITE * 3600 / SEG_POR_ITER)
cfg.agent.max_iterations = min(META, cabe_tempo)
print(f"envs        = {NUM_ENVS}")
print(f"lote do PPO = {NUM_ENVS * cfg.agent.num_steps_per_env} transições, "
      f"minilote {NUM_ENVS * cfg.agent.num_steps_per_env // cfg.agent.algorithm.num_mini_batches}")
print(f"lr          = {cfg.agent.algorithm.learning_rate} "
      f"({cfg.agent.algorithm.schedule})   seed {cfg.agent.seed}")
print(f"cabe em {HORAS_LIMITE:.1f} h a {SEG_POR_ITER:.1f} s/iter = {cabe_tempo}")
print(f"vai rodar   = {cfg.agent.max_iterations} de {META}")
if META > cabe_tempo:
    print(f"\n⚠ ESTA SESSÃO NÃO ALCANÇA A {META}. Ao terminar, rode a célula do "
          f"dataset e continue pelo notebook de RESUME.")
print(f"log_root    = {cfg.log_root}\n")

# ⚠ `try/finally`, e não a linha nua. Assim o pacote nasce também quando o treino
# estoura (OOM é o caso provável) ou quando você interrompe o kernel.
try:
    launch_training(g1_limpo.TASK_ID, cfg)
finally:
    empacota()

## Persistir o checkpoint fora da sessão

Sobe como **versão nova do dataset**. É a única persistência: `/kaggle/working` morre
com a sessão.

Os dois segredos precisam existir **e estar anexados** a este notebook
(Add-ons -> Secrets -> o toggle de cada um).

In [ ]:
import json, os, pathlib, shutil, subprocess

if ULTIMO_CKPT is None:
    empacota(baixa=False)
assert ULTIMO_CKPT is not None, "nada para subir: o treino não salvou checkpoint"

# ⚠⚠ DATASET PRÓPRIO, e NÃO o `g1-limpo-v2`. Aquele guarda o `model_14000`, o
# pipeline que funciona. O `kaggle datasets version` substitui o CONTEÚDO da
# versão: subir o zero por cima tiraria o checkpoint bom da versão mais recente,
# que é a que o `Add Input` dos outros notebooks enxerga.
# ⚠ VOCÊ TEM DE CRIAR ESTE DATASET ANTES, vazio, em kaggle.com/datasets.
SLUG = "g1-limpo-zero"      # o slug do SEU dataset DO ZERO

from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = _s.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = _s.get_secret("KAGGLE_KEY")
usuario = os.environ["KAGGLE_USERNAME"]

# ⚠ PASTA PRÓPRIA, com SÓ o que sobe. O `kaggle datasets version` envia o diretório
# INTEIRO — apontá-lo para /kaggle/working mandaria o clone e todos os checkpoints.
envio = pathlib.Path("/kaggle/working/envio")
shutil.rmtree(envio, ignore_errors=True)
envio.mkdir()
shutil.copy2(ULTIMO_CKPT, envio / ULTIMO_CKPT.name)
(envio / "dataset-metadata.json").write_text(json.dumps(
    {"title": "G1-Limpo-Zero", "id": f"{usuario}/{SLUG}",
     "licenses": [{"name": "CC0-1.0"}]}, indent=1))
print("vai subir:", sorted(p.name for p in envio.iterdir()))

r = subprocess.run(["kaggle", "datasets", "version", "-p", str(envio),
                    "-m", f"{RUN} it{ULTIMA_IT}", "-r", "skip"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, (
    "o upload falhou. Confira: os dois segredos anexados, a internet ligada, e o "
    f"dataset {usuario}/{SLUG} existindo e sendo seu.")
print(f"\nsubiu {ULTIMO_CKPT.name} em {usuario}/{SLUG}")
print("na próxima sessão: use o notebook de RESUME, que o acha pelo maior número")